In [ ]:
from pulp import *

model = LpProblem("optimasi_produksi_bakery", LpMaximize)

produk = [
    "Roti_Cokelat",
    "Roti_Keju",
    "Croissant_Cokelat",
    "Croissant_Keju"
]

minimum = 0

x = LpVariable.dicts("Produk", produk, lowBound = minimum)

Produk_Roti_Cokelat
Produk_Croissant_Keju


In [ ]:
bahan_kg = {
    "telur" : 32000,
    "tepung" : 15000,
    "coklat" : 12000,
    "keju" : 15000,
    "gula" : 20000,
    "mentega" : 15000
}

resep = {
    "roti_coklat": {
        "tepung": 1/8,
        "telur": 1,
        "coklat": 1/4,
        "gula": 1/10,
        "mentega": 1/12
    },

    "roti_keju": {
        "tepung": 1/8,
        "telur": 1,
        "keju": 1/5,
        "gula": 1/10,
        "mentega": 1/12
    },

    "croissant_coklat": {
        "tepung": 1/6,
        "telur": 2,
        "coklat": 1/3,
        "mentega": 1/5
    },

    "croissant_keju": {
        "tepung": 1/6,
        "telur": 2,
        "keju": 1/4,
        "mentega": 1/5
    }
}


In [12]:
biaya = {}
for produk, bahan in resep.items():
    total = 0

    for nama_bahan, jumlah in bahan.items():
        harga = bahan_kg[nama_bahan]
        print(f"produk : {produk} ; harga : {harga} ; bahan : {nama_bahan} ; jumlah : {jumlah} ; hasil : {jumlah * harga}")
        total += harga * jumlah
    print(produk, total)
    biaya[produk] = total

for barang, harga in biaya.items():
    print(barang, harga)

produk : roti_coklat ; harga : 15000 ; bahan : tepung ; jumlah : 0.125 ; hasil : 1875.0
produk : roti_coklat ; harga : 32000 ; bahan : telur ; jumlah : 1 ; hasil : 32000
produk : roti_coklat ; harga : 12000 ; bahan : coklat ; jumlah : 0.25 ; hasil : 3000.0
produk : roti_coklat ; harga : 20000 ; bahan : gula ; jumlah : 0.1 ; hasil : 2000.0
produk : roti_coklat ; harga : 15000 ; bahan : mentega ; jumlah : 0.08333333333333333 ; hasil : 1250.0
roti_coklat 40125.0
produk : roti_keju ; harga : 15000 ; bahan : tepung ; jumlah : 0.125 ; hasil : 1875.0
produk : roti_keju ; harga : 32000 ; bahan : telur ; jumlah : 1 ; hasil : 32000
produk : roti_keju ; harga : 15000 ; bahan : keju ; jumlah : 0.2 ; hasil : 3000.0
produk : roti_keju ; harga : 20000 ; bahan : gula ; jumlah : 0.1 ; hasil : 2000.0
produk : roti_keju ; harga : 15000 ; bahan : mentega ; jumlah : 0.08333333333333333 ; hasil : 1250.0
roti_keju 40125.0
produk : croissant_coklat ; harga : 15000 ; bahan : tepung ; jumlah : 0.166666666666666

In [21]:
harga_produk ={}
margin = 0.3
for i , j in biaya.items():
    harga_produk[i] = j + j*margin

print(harga_produk)

{'roti_coklat': 52162.5, 'roti_keju': 52162.5, 'croissant_coklat': 95550.0, 'croissant_keju': 95225.0}


In [22]:
import math
harga_produk_bulat = {}
for i, j in harga_produk.items():
    harga_produk_bulat[i] = math.ceil(j/1000) * 1000

print(harga_produk_bulat)

{'roti_coklat': 53000, 'roti_keju': 53000, 'croissant_coklat': 96000, 'croissant_keju': 96000}


In [ ]:
profit = {}

for i, j in harga_produk_bulat.items():
    profit[i] = j - biaya[i]

print(profit)

{'roti_coklat': 12875.0, 'roti_keju': 12875.0, 'croissant_coklat': 22500.0, 'croissant_keju': 22750.0}


In [25]:
from pulp import *

stok_bahan = {
    "telur" : 10,
    "tepung" : 15,
    "coklat" : 9,
    "keju" : 7,
    "gula" : 7,
    "mentega" : 10
}

model = LpProblem("optimasi_bakery", LpMaximize)

produk_list = list(resep.keys())
x = LpVariable.dicts(
    "Produksi",
    produk_list,
    lowBound=0,
    cat="Integer"
)

model += lpSum(
    profit[p] * x[p]
    for p in produk_list
)

for bahan in stok_bahan:

    model += lpSum(
        resep[produk].get(bahan, 0) * x[produk]
        for produk in produk_list
    ) <= stok_bahan[bahan]
model.solve()

print("Status:", LpStatus[model.status])

print("\nJumlah Produksi Optimal:")

for produk in produk_list:
    print(produk, "=", x[produk].varValue)

print("\nProfit Maksimum:")
print(value(model.objective))

Status: Optimal

Jumlah Produksi Optimal:
roti_coklat = 10.0
roti_keju = 0.0
croissant_coklat = 0.0
croissant_keju = 0.0

Profit Maksimum:
128750.0


In [7]:
import pandas as pd

url = "https://docs.google.com/spreadsheets/d/1rj8Ayd35jzFpiMnMvlftJU8MlwuZgbulH3zH7mQ6e9A/export?format=csv&gid=0"

df = pd.read_csv(url)
df

,nama_bahan,harga,stok
0,telur,32000,10
1,tepung,15000,15
2,coklat,12000,9
3,keju,15000,7
4,gula,20000,7
5,mentega,15000,10
